In [0]:
# Databricks notebook source
# MAGIC %md
# AI Data Ingestion: Project Setup
# Installs the library, creates the volume and folders, and validates the Gemini secret and client.

# COMMAND ----------

# 1: install lib
%pip install google-genai

# COMMAND ----------

# 2: restart python (must be in its own cell, it clears all Python variables)
dbutils.library.restartPython()

# COMMAND ----------

# 3: config (re-run after the restart, since variables are cleared)
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "AI_DataIngestion"
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

FOLDERS = [
    f"{BASE}/landing",
    f"{BASE}/data/processing",
    f"{BASE}/data/archive",
    f"{BASE}/data/error",
    f"{BASE}/data/output",
]

SECRET_SCOPE = "gemini"
SECRET_KEY = "GEMINI_API_KEY"

# COMMAND ----------

# 4: create the volume (dbutils has no volume command, so this uses spark.sql)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print(f"Volume ready: {BASE}")

# COMMAND ----------

# 5: create the folders with dbutils (safe to re-run, existing folders are left alone)
for folder in FOLDERS:
    dbutils.fs.mkdirs(folder)
    print(f"Created/exists: {folder}")

# COMMAND ----------

# 6: verify the structure
for item in dbutils.fs.ls(BASE):
    print(item.path)
    if item.name == "data/":
        for sub in dbutils.fs.ls(item.path):
            print("   ", sub.path)

# COMMAND ----------

# 7: check the secret scope and key exist
scopes = [s.name for s in dbutils.secrets.listScopes()]
assert SECRET_SCOPE in scopes, f"Secret scope '{SECRET_SCOPE}' not found. Available: {scopes}"

keys = [k.key for k in dbutils.secrets.list(SECRET_SCOPE)]
assert SECRET_KEY in keys, f"Key '{SECRET_KEY}' not found in scope. Available: {keys}"
print(f"Secret '{SECRET_KEY}' found in scope '{SECRET_SCOPE}'")

# COMMAND ----------

# 8: validate the secret without printing it
api_key = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
print(f"API key loaded, length = {len(api_key)}")

# COMMAND ----------

# 9: validate the Gemini client
from google import genai

client = genai.Client(api_key=api_key)
first_model = next(iter(client.models.list()))
print(f"Client OK. Example model available: {first_model.name}")

# COMMAND ----------

print("Project setup complete")